<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase42_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 42 — versioned loss-aware decision prototype, randomized synthetic challenges and rule ablations

**Colab → Runtime → Run all. No manual uploads.** This notebook mounts Google Drive, searches for the verified Phase 39 source ledger and Phase 41 outputs (including `(1)` filename suffixes and different folders), checks the SHA-256 manifest and all five Phase 41 companion CSVs, and writes a timestamped folder to `MyDrive/ICHI2027/Phase42/` after local staging and Drive read-back hashes.

**Scientific purpose:** Turn Phase 41's source-to-logical-projection detector into an explicitly versioned, *development-stage* decision algorithm. A **new, deterministically randomized, developer-authored synthetic fixture set** covers all 15 proxy-eligible people, the four proxy-ineligible people as clean controls, ten fault-injection families (person reassignment generates a paired study-link failure), combined faults, and valid cm³ normalization. Compare six **internal rule ablations** for event-level fault detection and policy-induced changes to *synthetic* cohort membership. Record both provenance/TRACE review decisions and critical-field exclusions.

**Not official comparative evidence:** No official OMOP/MI-CDM tables or ETL are used. The original generic TFL v15 (Arm C), official Arm A, real ADNI/MRI rows, and independently authored/blinded validation remain **unexecuted**. This is an engineering-development ablation, **not** A/B/C/D results or an unbiased sensitivity estimate. Every original TRACE state is PASS; REVIEW_REQUIRED cases are labeled fabricated fixtures. No clinical MCI labels or real DICOM UID are inferred. All ten Phase 41 Arm A blockers carry forward unchanged.

**Optional offline testing:** set `PHASE42_LOCAL_DRIVE_ROOT` to a Drive mirror; optional `PHASE42_DRIVE_FOLDER_HINT` restricts discovery. For personal Drive use, neither is needed.

In [9]:
# CODE CELL 1/8 — Mount Drive, define immutable inputs and local staging
import os, re, csv, json, hashlib, sqlite3, shutil, tempfile, zipfile, time, random
import datetime as dt
from pathlib import Path
from copy import deepcopy
from collections import Counter, defaultdict

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
EXPECTED_P39_MANIFEST='381734533579c739336b3d99dd221cf2de5b37b174743b7cbf7bfdc50201f0c8'
EXPECTED_P39_DB='0c9507d8fcf5986e9763d473bf0b342a1c4daaad6c87ee04915986518f5c2384'
EXPECTED_P41_GATES='4c1febebd83ee969f9102499ddd09fba6542092c202fcfd7d98015797f1f454b'
EXPECTED_P41_CONTRACT='d80e7ede309fedca3e95341b299ae4756c9a6796ea2148c4abc0479f8ee235da'
EXPECTED_P41_ARTIFACTS={
 'ICHI2027_Phase41_ARM_A_GATES_UNCHANGED.csv':'4c1febebd83ee969f9102499ddd09fba6542092c202fcfd7d98015797f1f454b',
 'ICHI2027_Phase41_CHALLENGE_CATALOG.csv':'6fdb81ebb3e7e7446b36fc2ea27534217560e88cc0e22f5f213f42868cb10a69',
 'ICHI2027_Phase41_COHORT_CONSEQUENCES.csv':'8729053e5cd7b5434f9ec5ede690619c4ac4594bace21e16167fb8b21c55a091',
 'ICHI2027_Phase41_DETECTION_EVENTS.csv':'da76a6669ac4de7e7cb85334636ea23737840087c888a339d0081a056bf3819e',
 'ICHI2027_Phase41_LOGICAL_CONTRACT_CARRIED_FORWARD.csv':'d80e7ede309fedca3e95341b299ae4756c9a6796ea2148c4abc0479f8ee235da',
}

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''): h.update(b)
    return h.hexdigest()

if os.environ.get('PHASE42_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE42_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
    print('Offline Drive-mirror mode:',MYDRIVE)
else:
    try: from google.colab import drive
    except ImportError as e: raise RuntimeError('Use Colab or set PHASE42_LOCAL_DRIVE_ROOT for offline testing.') from e
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir(), 'Drive mounted but MyDrive could not be read.'
    ROOTS=[MYDRIVE]
    shared=Path('/content/drive/Shareddrives')
    if shared.is_dir(): ROOTS.append(shared)
hint=os.environ.get('PHASE42_DRIVE_FOLDER_HINT','').strip()
if hint:
    p=Path(hint).expanduser()
    if not p.is_absolute(): p=MYDRIVE/p
    if not p.is_dir(): raise FileNotFoundError('Drive hint folder does not exist: '+hint)
    ROOTS=[p.resolve()]
WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase42_'))
INPUT=WORK/'inputs';INPUT.mkdir()
STAGE=WORK/'outputs';STAGE.mkdir()
OUT_BASE=Path(os.environ.get('PHASE42_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase42')))
OUT=OUT_BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
print('Drive search:',*[str(r) for r in ROOTS],sep='\n  ')
print('Output:',OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive search:
  /content/drive/MyDrive
Output: /content/drive/MyDrive/ICHI2027/Phase42/run_20260929T062745_568243Z


## 1 · Automatic, cryptographically checked source discovery

The notebook selects files by their scientific checksums, not by a potentially misleading Drive filename or timestamp. Exact, approved Phase 41 files can be anywhere under the searched Drive roots. A duplicated corrupted copy is ignored; if no checksum-matching file exists, processing stops. No file-upload dialog is used.

In [2]:
# CODE CELL 2/8 — Verify Phase 39 + Phase 41, including all five companion CSVs
REQUIRED={
 'p39_manifest':('ICHI2027_Phase39_SHAREABLE','.json'),
 'p39_db':('ICHI2027_Phase39_SYNTHETIC_SOURCE_LEDGER','.sqlite'),
 'p41_summary':('ICHI2027_Phase41_SHAREABLE','.json'),
}
for name in EXPECTED_P41_ARTIFACTS:
    REQUIRED[name]=(Path(name).stem,Path(name).suffix)

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext), actual,re.I) is not None

hits={k:[] for k in REQUIRED}
for root in ROOTS:
    for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
        dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
        for f in files:
            for key,(stem,ext) in REQUIRED.items():
                if name_matches(f,stem,ext):
                    hits[key].append(Path(here)/f); break
print('Drive candidates:',{k:len(v) for k,v in hits.items()})
missing=[k for k,v in hits.items() if not v]
if missing: raise FileNotFoundError('Missing required Drive files (no manual upload fallback): '+', '.join(missing))

def choose_hash(key,digest):
    good=[]
    for p in hits[key]:
        try:
            if sha256(p)==digest:good.append(p)
        except OSError:pass
    if not good: raise RuntimeError('Approved checksum not found for '+key+'. Expected SHA-256 '+digest)
    return sorted(good,key=str)[0]

P={}
P['p39_manifest']=choose_hash('p39_manifest',EXPECTED_P39_MANIFEST)
M39=json.loads(P['p39_manifest'].read_text('utf-8'))
assert M39['phase']=='39_sql_source_ledger_cohort_replay_and_integrity'
assert M39['artifact_sha256']['ICHI2027_Phase39_SYNTHETIC_SOURCE_LEDGER.sqlite']==EXPECTED_P39_DB
P['p39_db']=choose_hash('p39_db',EXPECTED_P39_DB)
for name,digest in EXPECTED_P41_ARTIFACTS.items():P[name]=choose_hash(name,digest)

valid=[]
for p in hits['p41_summary']:
    try:
        v=json.loads(p.read_text('utf-8'))
        fields={
          'project':PROJECT,'phase':'41_synthetic_E0_E7_challenge_and_cohort_consequences',
          'scope':'PUBLIC_SYNTHETIC_DEVELOPER_AUTHORED_ONLY',
          'phase39_manifest_sha256':EXPECTED_P39_MANIFEST,
          'phase39_immutable_sqlite_sha256':EXPECTED_P39_DB,
          'synthetic_people':19,'synthetic_studies':38,'synthetic_observations':148,
          'synthetic_source_proxy_positive':15,'clinical_mci_attested':False,
          'scenarios_executed':8,'developer_authored_injected_fault_events':9,
          'internal_detected_labeled_fault_events':9,'unexpected_detector_events':0,
          'valid_unit_normalizations_without_alert':1,
          'cohort_query_rows':32,'official_omop_etl_executed':False,
          'official_micdm_load_executed':False,'real_mri_rows_processed':0,
          'frozen_original_generic_tfl_armC_executed':False,
          'independently_blinded_external_validation':False,
          'full_A_B_C_D_evaluation':False,
          'armA_status':'BLOCKED_OFFICIAL_ARM_A_NOT_EXECUTED',
          'armA_unapproved_original_gates':10,
        }
        if all(v.get(k)==w for k,w in fields.items()) and all(v['artifact_sha256'].get(k)==w for k,w in EXPECTED_P41_ARTIFACTS.items()):
            valid.append((p,v))
    except (OSError,UnicodeError,ValueError,KeyError):pass
if not valid:raise RuntimeError('No Phase 41 shareable manifest matches the exact uploaded validated scientific results.')
P['p41_summary'],M41=sorted(valid,key=lambda x:(x[1].get('run_utc',''),str(x[0])))[-1]

with P['ICHI2027_Phase41_ARM_A_GATES_UNCHANGED.csv'].open(newline='',encoding='utf-8') as f:GATES=list(csv.DictReader(f))
with P['ICHI2027_Phase41_LOGICAL_CONTRACT_CARRIED_FORWARD.csv'].open(newline='',encoding='utf-8') as f:CONTRACT=list(csv.DictReader(f))
with P['ICHI2027_Phase41_CHALLENGE_CATALOG.csv'].open(newline='',encoding='utf-8') as f:CAT41=list(csv.DictReader(f))
with P['ICHI2027_Phase41_DETECTION_EVENTS.csv'].open(newline='',encoding='utf-8') as f:EVENTS41=list(csv.DictReader(f))
with P['ICHI2027_Phase41_COHORT_CONSEQUENCES.csv'].open(newline='',encoding='utf-8') as f:COHORT41=list(csv.DictReader(f))
assert len(GATES)==10 and all(r['official_validation_passed']=='False' for r in GATES)
assert len(CONTRACT)==8 and len(CAT41)==8 and len(EVENTS41)==9 and len(COHORT41)==32
assert sum(int(x['ground_truth_fault_events']) for x in CAT41)==9
assert {x['scenario'] for x in CAT41}=={'E'+str(i) for i in range(8)}
DB_LOCAL=INPUT/'immutable_source.sqlite'
shutil.copyfile(P['p39_db'],DB_LOCAL)
assert sha256(DB_LOCAL)==EXPECTED_P39_DB
print('PASS: Phase 39 manifest/SQLite and Phase 41 JSON + 5 CSVs independently hash-verified.')
print('Phase 41: 9/9 developer-injected events internally detected; original 10 Arm A gates still open.')

Drive candidates: {'p39_manifest': 1, 'p39_db': 1, 'p41_summary': 1, 'ICHI2027_Phase41_ARM_A_GATES_UNCHANGED.csv': 1, 'ICHI2027_Phase41_CHALLENGE_CATALOG.csv': 1, 'ICHI2027_Phase41_COHORT_CONSEQUENCES.csv': 1, 'ICHI2027_Phase41_DETECTION_EVENTS.csv': 1, 'ICHI2027_Phase41_LOGICAL_CONTRACT_CARRIED_FORWARD.csv': 1}
PASS: Phase 39 manifest/SQLite and Phase 41 JSON + 5 CSVs independently hash-verified.
Phase 41: 9/9 developer-injected events internally detected; original 10 Arm A gates still open.


## 2 · Load the immutable synthetic source and lock the internal experimental contract

Definitions and rules are fixed *in the code before running the randomized challenge*. This is **not preregistration or external blinding**. The policy matrix distinguishes critical measurement/linkage errors (EXCLUDE) from recoverable audit-evidence loss (REVIEW) and valid unit normalization (NORMALIZE). It never assumes missing TRACE state is PASS.

In [3]:
# CODE CELL 3/8 — Read-only synthetic source; documented, versioned internal policy
c=sqlite3.connect('file:'+str(DB_LOCAL)+'?mode=ro',uri=True)
assert c.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
assert c.execute('PRAGMA foreign_key_check').fetchall()==[]
c.row_factory=sqlite3.Row
STUDIES={r['study_ref']:dict(r) for r in c.execute('SELECT * FROM source_study')}
DEVICES={r['device_ref']:dict(r) for r in c.execute('SELECT * FROM source_device')}
PROVS={r['obs_ref']:dict(r) for r in c.execute('SELECT * FROM source_observation_provenance')}
SOURCE=[]
for raw in c.execute('SELECT * FROM source_observation ORDER BY obs_ref'):
    d=dict(raw);dev=DEVICES[d['device_ref']]
    d.update(algorithm_name=dev['algorithm_name'],algorithm_version=dev['algorithm_version'])
    assert PROVS[d['obs_ref']]['provenance_ref']==d['provenance_ref']
    SOURCE.append(d)
c.close()
assert len(SOURCE)==148 and len(STUDIES)==38 and len(DEVICES)==19
assert Counter(r['trace_state'] for r in SOURCE)=={'PASS':148}
assert not any(s['uid_source_attested'] for s in STUDIES.values())

RULE_GROUPS={
 'UNIT_ANATOMY':['ANATOMY_LATERALITY_MISMATCH','UNIT_MISSING_OR_INCOMPATIBLE','VALUE_MISMATCH'],
 'PAIRING':['PERSON_LINK_MISMATCH','STUDY_TIMEPOINT_LINK_MISMATCH'],
 'LINEAGE':['MODEL_IDENTITY_LOSS','PROVENANCE_LINK_LOSS'],
 'TRACE':['TRACE_STATE_LOSS'],
 'CARDINALITY':['DUPLICATE_OBSERVATION','DROPPED_OBSERVATION','UNEXPECTED_OBSERVATION'],
}
CRITICAL={'ANATOMY_LATERALITY_MISMATCH','UNIT_MISSING_OR_INCOMPATIBLE','VALUE_MISMATCH',
          'PERSON_LINK_MISMATCH','STUDY_TIMEPOINT_LINK_MISMATCH','DUPLICATE_OBSERVATION',
          'DROPPED_OBSERVATION','UNEXPECTED_OBSERVATION'}
REVIEW={'MODEL_IDENTITY_LOSS','PROVENANCE_LINK_LOSS','TRACE_STATE_LOSS'}
assert not CRITICAL&REVIEW
ABLATIONS={
 '00_VALUE_ONLY_DIAGNOSTIC':[],
 '01_UNIT_ANATOMY':['UNIT_ANATOMY'],
 '02_PLUS_PAIRING':['UNIT_ANATOMY','PAIRING'],
 '03_PLUS_LINEAGE':['UNIT_ANATOMY','PAIRING','LINEAGE'],
 '04_PLUS_TRACE':['UNIT_ANATOMY','PAIRING','LINEAGE','TRACE'],
 '05_FULL':['UNIT_ANATOMY','PAIRING','LINEAGE','TRACE','CARDINALITY'],
}
VERSION='phase42_internal_source_projection_v0.1'
SEED=420927
ABS_TOLERANCE_MM3=1e-6
METHOD={
 'method_version':VERSION,'scope':'DEVELOPER_SYNTHETIC_INTERNAL_ABLATION_NOT_ARM_D',
 'random_seed':SEED,'absolute_value_tolerance_mm3':ABS_TOLERANCE_MM3,
 'groups':RULE_GROUPS,'ablations':ABLATIONS,
 'critical_exclude_reason_codes':sorted(CRITICAL),'audit_review_reason_codes':sorted(REVIEW),
 'states':['PRESERVE','NORMALIZE','REVIEW','EXCLUDE'],
 'policy_notes':{
   'EXCLUDE':'Target record has unreconciled critical semantic/linkage/cardinality loss. No silent inclusion.',
   'REVIEW':'Value may remain in the unqualified proxy; qualified policy withholds audit-deficient record.',
   'NORMALIZE':'Valid, checked cm3 to mm3 conversion is not a fault.',
   'PRESERVE':'No error detected among currently active rules; not proof that all source evidence survived.',
 },
 'external_reviewer':False,'registered_before_external_validation':False,
 'official_target_schema_used':False,'frozen_original_TFL_v15_executed':False,
}
METHOD_BLOB=json.dumps(METHOD,sort_keys=True,separators=(',',':')).encode('utf-8')
METHOD_SHA256=hashlib.sha256(METHOD_BLOB).hexdigest()
print('Internal method:',VERSION,'SHA-256:',METHOD_SHA256)
print('Original source:',len(SOURCE),'rows; all original TRACE states PASS. No clinical MCI attested.')

Internal method: phase42_internal_source_projection_v0.1 SHA-256: 5bac736739cffa44b1ebbc2ced2886f7e267887a4e78a1b1ba8e012651616902
Original source: 148 rows; all original TRACE states PASS. No clinical MCI attested.


## 3 · Matched source and projection cohort SQL

The **reference** is the same frozen four-slot bilateral-hippocampal *synthetic proxy* used in Phase 41. The strict SQL verifies units, anatomical side, and the actual source study/date link; a separate qualified query also treats missing or fabricated REVIEW_REQUIRED TRACE as unqualified. A deliberately lossy value-only diagnostic is retained solely to demonstrate silent inclusion risks; **it is not Arm B**.

In [4]:
# CODE CELL 4/8 — Source/target synthetic SQL plus raw field-preserving projection
FIELDS=['obs_ref','person_ref','study_ref','biomarker_code','anatomy_source_code','laterality',
        'timepoint','effective_utc','value_mm3','unit_code','algorithm_name','algorithm_version',
        'trace_state','trace_version','device_ref','provenance_ref']
DDL="""
CREATE TABLE projected_study(study_ref TEXT,person_ref TEXT,study_started TEXT);
CREATE TABLE projected_feature(obs_ref TEXT,person_ref TEXT,study_ref TEXT,
 biomarker_code TEXT,anatomy_source_code TEXT,laterality TEXT,timepoint TEXT,
 effective_utc TEXT,value_mm3 REAL,unit_code TEXT,algorithm_name TEXT,
 algorithm_version TEXT,trace_state TEXT,trace_version TEXT,device_ref TEXT,provenance_ref TEXT);
"""
INSERT='INSERT INTO projected_feature('+','.join(FIELDS)+') VALUES('+','.join('?' for _ in FIELDS)+')'
SQL_PROXY="""
WITH observed AS(
 SELECT f.*,s.person_ref AS linked_person,s.study_started AS linked_started,
 CASE WHEN f.unit_code='mm3' THEN f.value_mm3
 WHEN f.unit_code='cm3' THEN f.value_mm3*1000.0 ELSE NULL END AS normalized_mm3
 FROM projected_feature f LEFT JOIN projected_study s ON f.study_ref=s.study_ref
 WHERE f.biomarker_code='hippocampus-volume'
),valid AS (
 SELECT *,CASE WHEN linked_person=person_ref AND linked_started=effective_utc
 AND normalized_mm3>0 AND provenance_ref IS NOT NULL
 AND ((laterality='LEFT' AND anatomy_source_code='SYN-HIP-LEFT')
 OR (laterality='RIGHT' AND anatomy_source_code='SYN-HIP-RIGHT'))
 AND timepoint IN ('baseline','12-month') THEN 1 ELSE 0 END AS valid_flag
 FROM observed
),slots AS(
 SELECT person_ref,timepoint,laterality,COUNT(*) AS n,SUM(valid_flag) AS nvalid,
 SUM(CASE WHEN trace_state IS NULL OR trace_state='REVIEW_REQUIRED' THEN 1 ELSE 0 END) AS unqualified
 FROM valid GROUP BY person_ref,timepoint,laterality
),eligible AS (
 SELECT person_ref FROM slots GROUP BY person_ref
 HAVING COUNT(*)=4 AND MIN(n)=1 AND MAX(n)=1 AND SUM(nvalid)=4
)
SELECT person_ref FROM eligible ORDER BY person_ref
"""
SQL_QUALIFIED=SQL_PROXY.replace('AND SUM(nvalid)=4','AND SUM(nvalid)=4 AND SUM(unqualified)=0')
SQL_VALUE_ONLY="""
SELECT person_ref FROM projected_feature WHERE biomarker_code='hippocampus-volume'
AND value_mm3>0 AND laterality IN ('LEFT','RIGHT') AND timepoint IN ('baseline','12-month')
GROUP BY person_ref HAVING COUNT(DISTINCT timepoint||'/'||laterality)=4
ORDER BY person_ref
"""

def query(rows,sql):
    cx=sqlite3.connect(':memory:')
    try:
        cx.executescript(DDL)
        cx.executemany('INSERT INTO projected_study VALUES (?,?,?)',
                       [(s['study_ref'],s['person_ref'],s['study_started']) for s in STUDIES.values()])
        cx.executemany(INSERT,[tuple(r.get(k) for k in FIELDS) for r in rows])
        return {z[0] for z in cx.execute(sql)}
    finally:cx.close()

SOURCE_PROXY=query(SOURCE,SQL_PROXY)
SOURCE_QUAL=query(SOURCE,SQL_QUALIFIED)
assert len(SOURCE_PROXY)==15 and SOURCE_QUAL==SOURCE_PROXY
ALL_PEOPLE=sorted({r['person_ref'] for r in SOURCE})
ELIGIBLE=sorted(SOURCE_PROXY)
INELIGIBLE=sorted(set(ALL_PEOPLE)-SOURCE_PROXY)
assert len(ALL_PEOPLE)==19 and len(INELIGIBLE)==4
HIP={p:[r['obs_ref'] for r in SOURCE if r['person_ref']==p and r['biomarker_code']=='hippocampus-volume'] for p in ALL_PEOPLE}
assert all(len(HIP[p])==4 for p in ELIGIBLE)
print('Frozen synthetic reference: 19 people, 15 proxy eligible, 4 proxy ineligible.')

Frozen synthetic reference: 19 people, 15 proxy eligible, 4 proxy ineligible.


## 4 · Generate NEW seeded development fixtures (not an independent hold-out)

For each of the 15 eligible synthetic people, generate ten single-fault families, one two-fault combination, and one valid normalization; add clean controls for all 19 people. Source records remain immutable. Family and row selection are seeded for exact replay but **all faults and their labels are still developer-authored**. Unlike Phase 41's small E0–E7 catalog, this varies the target person and four possible hippocampal acquisition slots and includes missing rows, person-link errors, and numerical mismatches.

In [5]:
# CODE CELL 5/8 — Seeded fixture generator; injector truth never passed to detector
RNG=random.Random(SEED)
FAMILIES=[
 'ANATOMY_LATERALITY_MISMATCH','UNIT_MISSING_OR_INCOMPATIBLE',
 'STUDY_TIMEPOINT_LINK_MISMATCH','MODEL_IDENTITY_LOSS','TRACE_STATE_LOSS',
 'DUPLICATE_OBSERVATION','PROVENANCE_LINK_LOSS','DROPPED_OBSERVATION',
 'PERSON_LINK_MISMATCH','VALUE_MISMATCH',
]
FIXTURES=[]


def new_case(kind,person,main_obs=None,secondary_obs=None):
    f={'case_id':f'case_{len(FIXTURES):04d}','kind':kind,'person_ref':person,
       'reference':deepcopy(SOURCE),'projected':deepcopy(SOURCE),
       'truth':set(),'normalization_expected':set(),
       'main_obs':main_obs or '', 'secondary_obs':secondary_obs or ''}
    FIXTURES.append(f)
    return f

def row_by_id(rows,ref):
    hit=[r for r in rows if r['obs_ref']==ref]
    assert len(hit)==1,ref
    return hit[0]

def inject(f,kind,obs):
    if kind=='DROPPED_OBSERVATION':
        f['projected']=[r for r in f['projected'] if r['obs_ref']!=obs]
    else:
        t=row_by_id(f['projected'],obs)
        if kind=='ANATOMY_LATERALITY_MISMATCH':
            t['laterality']='LEFT' if t['laterality']=='RIGHT' else 'RIGHT'
        elif kind=='UNIT_MISSING_OR_INCOMPATIBLE':t['unit_code']=None
        elif kind=='STUDY_TIMEPOINT_LINK_MISMATCH':
            t['study_ref']=next(s['study_ref'] for s in STUDIES.values()
                                if s['person_ref']==t['person_ref'] and s['study_started']!=t['effective_utc'])
        elif kind=='MODEL_IDENTITY_LOSS':t['algorithm_version']=None
        elif kind=='TRACE_STATE_LOSS':
            # Test-only source-state overlay; original immutable Phase 39 source stays PASS.
            row_by_id(f['reference'],obs)['trace_state']='REVIEW_REQUIRED'
            t['trace_state']=None
        elif kind=='DUPLICATE_OBSERVATION':f['projected'].append(deepcopy(t))
        elif kind=='PROVENANCE_LINK_LOSS':t['provenance_ref']=None
        elif kind=='PERSON_LINK_MISMATCH':
            t['person_ref']=next(p for p in ALL_PEOPLE if p!=t['person_ref'])
            # Changing the person also breaks the linked source-study/person relation: two observable fault codes.
            f['truth'].add((obs,'STUDY_TIMEPOINT_LINK_MISMATCH'))
        elif kind=='VALUE_MISMATCH':t['value_mm3']=float(t['value_mm3'])+20.0
        else:raise AssertionError('Unexpected fixture family '+kind)
    f['truth'].add((obs,kind))

for person in ALL_PEOPLE:
    new_case('CLEAN',person)
for person in ELIGIBLE:
    for kind in FAMILIES:
        obs=RNG.choice(HIP[person]);f=new_case(kind,person,obs)
        inject(f,kind,obs)
    obs=RNG.choice(HIP[person]);f=new_case('VALID_CM3_NORMALIZATION',person,obs)
    t=row_by_id(f['projected'],obs)
    t['unit_code']='cm3';t['value_mm3']=float(t['value_mm3'])/1000.0
    f['normalization_expected'].add(obs)
    refs=RNG.sample(HIP[person],2)
    f=new_case('COMBINED_UNIT_AND_LINEAGE',person,refs[0],refs[1])
    inject(f,'UNIT_MISSING_OR_INCOMPATIBLE',refs[0])
    inject(f,'PROVENANCE_LINK_LOSS',refs[1])
assert len(FIXTURES)==19+15*(len(FAMILIES)+2)
assert len(FIXTURES)==199
assert sum(len(f['truth']) for f in FIXTURES)==195
assert sum(len(f['normalization_expected']) for f in FIXTURES)==15
print('Prepared',len(FIXTURES),'synthetic cases with',195,'injected events, 15 valid conversions, 19 clean cases.')

Prepared 199 synthetic cases with 195 injected events, 15 valid conversions, 19 clean cases.


## 5 · Execute the source-to-projection detector and explicit loss-aware policy

The detector receives source/reference and projected records, **not injected truth labels**. It checks exact observation cardinality; source participant/study and timepoint links; region and laterality; numerical values after known mm³/cm³ conversion; model identity; provenance; and TRACE state. The policy separately records what each rule ablation can see. `PRESERVE` under an ablation means only *no active-rule alert*, not that the record is medically valid.

In [6]:
# CODE CELL 6/8 — Detector; label-separated audit and policy, no official target ETL

def detect(ref,target):
    src={x['obs_ref']:x for x in ref}
    assert len(src)==len(ref),'Fixture reference contains duplicate IDs'
    tgt=defaultdict(list)
    for r in target:tgt[r['obs_ref']].append(r)
    alerts=set(); normalized=set()
    for obs,s in src.items():
        got=tgt.get(obs,[])
        if len(got)!=1:
            alerts.add((obs,'DUPLICATE_OBSERVATION' if len(got)>1 else 'DROPPED_OBSERVATION'))
            if not got:continue
        t=got[0]
        if t['person_ref']!=s['person_ref']:alerts.add((obs,'PERSON_LINK_MISMATCH'))
        if any(t[k]!=s[k] for k in ('biomarker_code','anatomy_source_code','laterality')):
            alerts.add((obs,'ANATOMY_LATERALITY_MISMATCH'))
        if (any(t[k]!=s[k] for k in ('study_ref','timepoint','effective_utc'))
            or t['study_ref'] not in STUDIES
            or STUDIES[t['study_ref']]['person_ref']!=t['person_ref']
            or STUDIES[t['study_ref']]['study_started']!=t['effective_utc']):
            alerts.add((obs,'STUDY_TIMEPOINT_LINK_MISMATCH'))
        unit=t['unit_code'];v=t['value_mm3']
        if unit not in {'mm3','cm3'} or not isinstance(v,(int,float)) or v<=0:
            alerts.add((obs,'UNIT_MISSING_OR_INCOMPATIBLE'))
        elif abs(float(v)*(1000.0 if unit=='cm3' else 1.0)-float(s['value_mm3']))>ABS_TOLERANCE_MM3:
            alerts.add((obs,'VALUE_MISMATCH'))
        elif unit=='cm3' and s['unit_code']=='mm3':normalized.add(obs)
        if t['algorithm_name']!=s['algorithm_name'] or t['algorithm_version']!=s['algorithm_version']:
            alerts.add((obs,'MODEL_IDENTITY_LOSS'))
        if t['trace_state']!=s['trace_state'] or t['trace_version']!=s['trace_version']:
            alerts.add((obs,'TRACE_STATE_LOSS'))
        if t['provenance_ref']!=s['provenance_ref'] or t['device_ref']!=s['device_ref']:
            alerts.add((obs,'PROVENANCE_LINK_LOSS'))
    for obs in set(tgt)-set(src):alerts.add((obs,'UNEXPECTED_OBSERVATION'))
    return alerts,normalized

def active_codes(groups):
    return {code for g in groups for code in RULE_GROUPS[g]}

def decisions(events,normalizations,enabled):
    per=defaultdict(set)
    for obs,code in events:
        if code in enabled:per[obs].add(code)
    out={}
    for obs,reasons in per.items():
        if reasons&CRITICAL: out[obs]='EXCLUDE'
        elif reasons&REVIEW:out[obs]='REVIEW'
        else:raise AssertionError('Unclassified reason code: '+str(reasons))
    for obs in normalizations:
        if obs not in out and 'UNIT_MISSING_OR_INCOMPATIBLE' in enabled:out[obs]='NORMALIZE'
    return out

REGRESSION=[]
for f in FIXTURES:
    alerts,norm=detect(f['reference'],f['projected'])
    # Engineering assertion only; labels come from the same project's authored generator.
    assert alerts==f['truth'],f['case_id']+' discrepancy: '+str((alerts-f['truth'],f['truth']-alerts))
    assert norm==f['normalization_expected'],f['case_id']+' conversion discrepancy'
    f['detected']=alerts;f['normalized']=norm
    REGRESSION.append({'case_id':f['case_id'],'kind':f['kind'],'person_ref':f['person_ref'],
      'main_synthetic_obs_ref':f['main_obs'],'second_synthetic_obs_ref':f['secondary_obs'],
      'projected_rows':len(f['projected']),'injected_truth_events':len(f['truth']),
      'detected_events':len(alerts),'false_alerts':len(alerts-f['truth']),
      'normalized_no_alert':len(norm),'same_developer_labels':'True'})
assert not detect(SOURCE,SOURCE)[0]
print('Internal source-to-projection regression PASS: 199 cases; all 195 authored faults recognized;')
print('15 valid unit conversions and 19 clean controls generated no alerts. Not independent validation.')

Internal source-to-projection regression PASS: 199 cases; all 195 authored faults recognized;
15 valid unit conversions and 19 clean controls generated no alerts. Not independent validation.


## 6 · Internal ablation and downstream synthetic cohort membership

Each of six ablations uses the same 199 synthetic cases. Report **event-level exact matches** and false alerts against *author-created* labels, separately from **person-level cohort membership**. A detection-based `EXCLUDE` deliberately withholds a damaged observation rather than silently treating it as valid; this can increase false exclusions versus an intact source cohort. A `REVIEW` withholds an observation only from the qualified-evidence query. Thus the algorithm is not presumed to maximize raw cohort retention.

In [7]:
# CODE CELL 7/8 — Ablation tables, explicit policies and matched cohort comparisons
METRICS=[];POLICY_COHORT=[];RULE_COVERAGE=[]

def cmp(a,b):
    u=a|b
    return {'source_n':len(a),'target_n':len(b),
            'false_inclusions':len(b-a),'false_exclusions':len(a-b),
            'jaccard':round(len(a&b)/len(u),6) if u else 1.0}

for ablation,groups in ABLATIONS.items():
    enabled=active_codes(groups)
    gt=tp=fp=0
    byfamily=defaultdict(lambda:Counter())
    state_counter=Counter()
    for f in FIXTURES:
        selected={(obs,c) for obs,c in f['detected'] if c in enabled}
        truth=f['truth']
        gt+=len(truth);tp+=len(selected&truth);fp+=len(selected-truth)
        for obs,c in truth:
            byfamily[c]['truth']+=1
            if (obs,c) in selected:byfamily[c]['matched']+=1
        for obs,c in selected-truth:byfamily[c]['false_alerts']+=1
        action=decisions(f['detected'],f['normalized'],enabled)
        state_counter.update(action.values())
        # Logical projection is separate from original source; no writes to source SQLite.
        unqualified_rows=[x for x in f['projected'] if action.get(x['obs_ref'])!='EXCLUDE']
        qualified_rows=[x for x in unqualified_rows if action.get(x['obs_ref'])!='REVIEW']
        ref_proxy=query(f['reference'],SQL_PROXY)
        ref_qual=query(f['reference'],SQL_QUALIFIED)
        assert len(ref_proxy)==15
        projected_strict=query(f['projected'],SQL_PROXY)
        diagnostic_value=query(f['projected'],SQL_VALUE_ONLY)
        policy_proxy=query(unqualified_rows,SQL_VALUE_ONLY)
        policy_qual=query(qualified_rows,SQL_VALUE_ONLY)
        # The diagnostic is deliberately permissive; policy filters are explicit loss decisions.
        outcomes=[
          ('STRICT_PROJECTED_DIAGNOSTIC',ref_proxy,projected_strict),
          ('VALUE_ONLY_UNSAFE_DIAGNOSTIC',ref_proxy,diagnostic_value),
          ('POLICY_PROXY_VALUE_ONLY_BASE',ref_proxy,policy_proxy),
          ('POLICY_QUALIFIED_VALUE_ONLY_BASE',ref_qual,policy_qual),
        ]
        for label,ref,tgt in outcomes:
            POLICY_COHORT.append({'ablation':ablation,'case_id':f['case_id'],
              'fault_family':f['kind'],'cohort_policy':label,
              'source_people_total':19,'source_rows_total':148,
              **cmp(ref,tgt), 'original_MCI_attested':'False','official_target':'False'})
    METRICS.append({'ablation':ablation,'active_groups':';'.join(groups) if groups else 'NONE',
      'active_reason_codes':len(enabled),'synthetic_cases':len(FIXTURES),
      'injected_fault_events':gt,'internally_matched_fault_events':tp,
      'missed_injected_events':gt-tp,'false_alert_events':fp,
      'internal_event_recall':round(tp/gt,6) if gt else '',
      'internal_event_precision':round(tp/(tp+fp),6) if tp+fp else '',
      'exclude_actions':state_counter['EXCLUDE'],'review_actions':state_counter['REVIEW'],
      'normalization_actions':state_counter['NORMALIZE'],
      'not_independent_scientific_estimate':'True'})
    for family in FAMILIES:
        x=byfamily[family]
        RULE_COVERAGE.append({'ablation':ablation,'family':family,
          'authored_truth_events':x['truth'],'internal_matched_events':x['matched'],
          'missed_events':x['truth']-x['matched'],
          'false_alert_events':x['false_alerts'],
          'enabled':'True' if family in enabled else 'False'})

assert len(METRICS)==6 and len(RULE_COVERAGE)==60
assert len(POLICY_COHORT)==6*len(FIXTURES)*4
assert METRICS[0]['internally_matched_fault_events']==0
assert METRICS[-1]['internally_matched_fault_events']==195
assert METRICS[-1]['false_alert_events']==0
assert sum(int(x['injected_truth_events']) for x in REGRESSION)==195
assert len(GATES)==10 and all(g['official_validation_passed']=='False' for g in GATES)
for r in METRICS:
    print(r['ablation'],':',r['internally_matched_fault_events'],'of',r['injected_fault_events'],
          'injected events visible; exclude',r['exclude_actions'],'review',r['review_actions'])
print('PASS: 6 ablations,',len(RULE_COVERAGE),'per-family rows,',len(POLICY_COHORT),'paired cohort comparisons.')

00_VALUE_ONLY_DIAGNOSTIC : 0 of 195 injected events visible; exclude 0 review 0
01_UNIT_ANATOMY : 60 of 195 injected events visible; exclude 60 review 0
02_PLUS_PAIRING : 105 of 195 injected events visible; exclude 90 review 0
03_PLUS_LINEAGE : 150 of 195 injected events visible; exclude 90 review 45
04_PLUS_TRACE : 165 of 195 injected events visible; exclude 90 review 60
05_FULL : 195 of 195 injected events visible; exclude 120 review 60
PASS: 6 ablations, 60 per-family rows, 4776 paired cohort comparisons.


## 7 · Save public-safe scientific-development artifacts back to Drive

All generated files contain only the publicly synthetic IDs from Phase 39 or aggregate counts. Writes happen on local Colab disk first. The notebook then saves each file and ZIP to a new timestamped Drive folder, checking the read-back SHA-256. No private Athena files, participant records, clinical identifiers, or unauthorized medical imaging exports are read.

In [8]:
# CODE CELL 8/8 — Build local outputs, manifest + ZIP; copy and verify Drive read-back hashes

def save_csv(name,rows):
    assert rows
    p=STAGE/name
    with p.open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]))
        writer.writeheader();writer.writerows(rows)
    return p

results=[
 save_csv('ICHI2027_Phase42_FIXTURE_MANIFEST.csv',REGRESSION),
 save_csv('ICHI2027_Phase42_ABLATION_METRICS.csv',METRICS),
 save_csv('ICHI2027_Phase42_RULE_COVERAGE.csv',RULE_COVERAGE),
 save_csv('ICHI2027_Phase42_POLICY_COHORT.csv',POLICY_COHORT),
 save_csv('ICHI2027_Phase42_ARM_A_GATES_UNCHANGED.csv',GATES),
 save_csv('ICHI2027_Phase42_LOGICAL_CONTRACT_UNCHANGED.csv',CONTRACT),
]
method_path=STAGE/'ICHI2027_Phase42_METHOD_SPEC.json'
method_path.write_text(json.dumps(METHOD,sort_keys=True,indent=2)+'\n',encoding='utf-8')
results.append(method_path)
readme=STAGE/'ICHI2027_Phase42_README.txt'
readme.write_text(
 'PHASE 42: SOURCE-TO-LOGICAL-PROJECTION, INTERNAL DEVELOPER-AUTHORED SYNTHETIC ONLY.\n'
 'Nothing in this folder establishes official OMOP or MI-CDM loading, frozen TFL v15, or clinical validity.\n'
 'No ADNI subject IDs, real patient data, real DICOM UID, Athena snapshot, or reviewed concept IDs were used.\n'
 'All 148 ORIGINAL TRACE states were PASS. Injected REVIEW_REQUIRED overlays are artificial test data.\n'
 'The source synthetic proxy cohort is 15 of 19 and is NOT MCI-diagnosed.\n'
 'Event match metrics reflect developer-known injected labels; no external blinding or independent validation.\n'
 'Value-only diagnostic is NOT Arm B; policy ablations are NOT Arm A/B/C/D results.\n'
 'Critical faults EXCLUDE; audit/TRACE loss REVIEW; known cm3/mm3 normalization yields NORMALIZE.\n'
 'Potential cohort false exclusions after safe EXCLUDE are reported, not hidden.\n'
 'Official Arm A remains blocked on the same 10 Phase 41 gates.\n',encoding='utf-8')
results.append(readme)
summary={
 'project':PROJECT,'phase':'42_internal_synthetic_policy_algorithm_and_ablations',
 'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'scope':'PUBLIC_SYNTHETIC_DEVELOPER_AUTHORED_ONLY',
 'input_phase39_manifest_sha256':sha256(P['p39_manifest']),
 'input_phase39_db_sha256':sha256(DB_LOCAL),
 'input_phase41_summary_sha256':sha256(P['p41_summary']),
 'input_phase41_companion_sha256':{k:sha256(P[k]) for k in EXPECTED_P41_ARTIFACTS},
 'method_version':VERSION,'method_config_sha256':METHOD_SHA256,
 'synthetic_people':19,'synthetic_source_observations':148,
 'synthetic_proxy_positive':15,'synthetic_proxy_negative':4,
 'seeded_developer_generated_cases':len(FIXTURES),
 'synthetic_injected_events':195,'internally_matched_events_full_rules':METRICS[-1]['internally_matched_fault_events'],
 'unexpected_alert_events_full_rules':METRICS[-1]['false_alert_events'],
 'normalization_without_alert_cases':15,
 'internal_ablations':len(ABLATIONS),
 'per_family_ablation_rows':len(RULE_COVERAGE),
 'person_level_policy_cohort_comparisons':len(POLICY_COHORT),
 'source_clinical_mci_attested':False,'source_original_trace_states':{'PASS':148},
 'synthetic_trace_review_overlays_only':True,
 'independently_blinded_external_validation':False,
 'official_omop_etl_executed':False,'official_micdm_load_executed':False,
 'frozen_original_generic_tfl_armC_executed':False,
 'real_mri_rows_processed':0,'full_four_arm_evaluation':False,
 'armA_status':'BLOCKED_OFFICIAL_ARM_A_NOT_EXECUTED','armA_unapproved_original_gates':len(GATES),
 'artifact_sha256':{p.name:sha256(p) for p in results},
 'warning':'Developer-designed synthetic ablations and a source-to-projection comparison algorithm are not an official ETL, a frozen standard comparator, or independently blinded evidence.'
}
summary_path=STAGE/'ICHI2027_Phase42_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
public=results+[summary_path]
zip_path=STAGE/'ICHI2027_Phase42_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in public:z.write(p,p.name)
assert len(zipfile.ZipFile(zip_path).namelist())==len(public)
OUT.mkdir(parents=True,exist_ok=False)
for p in public+[zip_path]:
    target=OUT/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,target)
            assert sha256(target)==sha256(p),'Drive read-back SHA-256 mismatch'
            break
        except (OSError,AssertionError) as e:
            if attempt==2: raise RuntimeError('Phase 42 generated locally, but saving to Drive failed: '+str(p)) from e
            print('Retrying Drive save:',p.name,'attempt',attempt+1)
            time.sleep(1+attempt)
print('SUCCESS:',len(public)+1,'Phase 42 public synthetic files written and read-back SHA-256 verified.')
print('Google Drive output folder:',OUT)
print('Shareable JSON:',OUT/summary_path.name)
print('Public ZIP:',OUT/zip_path.name)

SUCCESS: 10 Phase 42 public synthetic files written and read-back SHA-256 verified.
Google Drive output folder: /content/drive/MyDrive/ICHI2027/Phase42/run_20260929T062140_462509Z
Shareable JSON: /content/drive/MyDrive/ICHI2027/Phase42/run_20260929T062140_462509Z/ICHI2027_Phase42_SHAREABLE.json
Public ZIP: /content/drive/MyDrive/ICHI2027/Phase42/run_20260929T062140_462509Z/ICHI2027_Phase42_PUBLIC_OUTPUTS.zip


## The **real** next research gate

An internal detector that rechecks source-to-projection equality can be useful engineering infrastructure, but it is **not yet** a demonstrated new clinical algorithm or a completed comparison with existing ETL. For official Arm A, obtain the authorized frozen Athena snapshot and valid manifest; independent adjudication for all 19 concept bindings and 36 original vocabulary-gap instances; approval of source-absent demographic, Procedure, and real DICOM UID policies; and the exact approved OMOP/MI-CDM revision with DDL. Then execute an official target loader with FK, cardinality, replay, and matched SQL tests. Run the **unmodified, frozen generic TFL v15** as a genuine later comparator; seek independently authored and blinded challenge cases for eventual scientific claims. Do not publish the internal 195/195 developer-label regression as a sensitivity estimate.